# RepOpt-3D Colab Fusion Eval

This notebook bootstraps a Colab runtime for OpenScene fusion-mode evaluation without installing `MinkowskiEngine`.

Expected runtime:
- Colab with GPU enabled
- repo branch: `dev-khalit-reopt3d`
- OpenScene submodule branch: `dev-khalit-openscene`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Fill these in before running the notebook.
REPO_URL = "https://github.com/a1bhinav/repopt-3d.git"
REPO_BRANCH = "dev-khalit-reopt3d"

OPENSCENE_FORK_URL = "https://github.com/St1p42/openscene.git"
OPENSCENE_BRANCH = "dev-khalit-openscene"

# Drive-backed storage roots.
DRIVE_OPENSCENE_DATA_ROOT = "/content/drive/MyDrive/repopt-data/openscene_data"
SAVE_ROOT = "/content/drive/MyDrive/repopt-output/fusion_eval"

# Data bootstrap behavior.
DOWNLOAD_IF_MISSING = True
BUILD_ONE_SCENE_SUBSET = True
SCENE_NAME = None  # Set explicitly, or leave as None to auto-pick the first available test scene.

# Set to 1 for the first debug run.
TEST_REPEATS = 1


In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!rm -rf repopt-3d
!git clone --recurse-submodules {REPO_URL}
%cd /content/repopt-3d
!git checkout {REPO_BRANCH}
!git submodule update --init --recursive
%cd /content/repopt-3d/third_party/openscene
!git remote set-url origin {OPENSCENE_FORK_URL}
!git fetch origin {OPENSCENE_BRANCH}
!git checkout {OPENSCENE_BRANCH}
%cd /content/repopt-3d


In [ ]:
import os
import shutil
from pathlib import Path

OPENSCENE_DIR = Path('/content/repopt-3d/third_party/openscene')
DRIVE_DATA_DIR = Path(DRIVE_OPENSCENE_DATA_ROOT)
DATA_SYMLINK = OPENSCENE_DIR / 'data'

DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)
create_symlink = True
if DATA_SYMLINK.is_symlink() or DATA_SYMLINK.exists():
    if DATA_SYMLINK.is_symlink() and DATA_SYMLINK.resolve() == DRIVE_DATA_DIR:
        create_symlink = False
    else:
        if DATA_SYMLINK.is_symlink() or DATA_SYMLINK.is_file():
            DATA_SYMLINK.unlink()
        else:
            shutil.rmtree(DATA_SYMLINK)
if create_symlink:
    os.symlink(DRIVE_DATA_DIR, DATA_SYMLINK)
print('OpenScene data symlink ->', DATA_SYMLINK.resolve())


In [ ]:
!python -V
!pip install --upgrade pip
!pip install --no-cache-dir torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu124
!pip install --no-cache-dir scipy open3d ftfy tensorboardx tqdm imageio plyfile opencv-python sharedarray git+https://github.com/openai/CLIP.git


In [ ]:
!python -c "import scipy, open3d, ftfy, tensorboardX, tqdm, imageio, plyfile, cv2, SharedArray, clip; print('All OpenScene deps OK')"

In [ ]:
# Standalone long-running cell: safe to rerun after interruption.
import subprocess
from pathlib import Path

OPENSCENE_DIR = Path('/content/repopt-3d/third_party/openscene')
DRIVE_DATA_DIR = Path(DRIVE_OPENSCENE_DATA_ROOT)
DATASET_ROOT = Path(DRIVE_OPENSCENE_DATA_ROOT) / 'matterport_3d'
DATASET_TEST_ROOT = DATASET_ROOT / 'test'
FUSED_ROOT_FULL = Path(DRIVE_OPENSCENE_DATA_ROOT) / 'matterport_multiview_openseg_test'

has_3d = DATASET_TEST_ROOT.exists() and any(DATASET_TEST_ROOT.glob('*.pth'))
has_fused = FUSED_ROOT_FULL.exists() and any(FUSED_ROOT_FULL.glob('*.pt'))

print('matterport_3d present =', has_3d)
print('matterport_multiview_openseg_test present =', has_fused)

if DOWNLOAD_IF_MISSING and not has_3d:
    subprocess.run("printf '2\n' | bash scripts/download_dataset.sh", shell=True, check=True, cwd=OPENSCENE_DIR)
if DOWNLOAD_IF_MISSING and not has_fused:
    subprocess.run("printf '3\n' | bash scripts/download_fused_features.sh", shell=True, check=True, cwd=OPENSCENE_DIR)

for zip_name in ['matterport_3d.zip', 'matterport_multiview_openseg_test.zip']:
    zip_path = Path(DRIVE_OPENSCENE_DATA_ROOT) / zip_name
    if zip_path.exists():
        zip_path.unlink()
        print('Deleted', zip_path)

num_3d = len(list(DATASET_TEST_ROOT.glob('*.pth'))) if DATASET_TEST_ROOT.exists() else 0
num_fused = len(list(FUSED_ROOT_FULL.glob('*.pt'))) if FUSED_ROOT_FULL.exists() else 0
print('test scene files =', num_3d)
print('fused feature files =', num_fused)


In [ ]:
# Standalone long-running cell: safe to rerun after interruption.
import os
import shutil
from pathlib import Path

DRIVE_DATA_DIR = Path(DRIVE_OPENSCENE_DATA_ROOT)
DATASET_ROOT = DRIVE_DATA_DIR / 'matterport_3d'
DATASET_TEST_ROOT = DATASET_ROOT / 'test'
FUSED_ROOT_FULL = DRIVE_DATA_DIR / 'matterport_multiview_openseg_test'
DATA_ROOT = str(DATASET_ROOT)
FUSED_ROOT = str(FUSED_ROOT_FULL)

if BUILD_ONE_SCENE_SUBSET:
    available_scenes = sorted(p.stem for p in DATASET_TEST_ROOT.glob('*.pth'))
    if not available_scenes:
        raise RuntimeError('No test scenes found under matterport_3d/test')
    if SCENE_NAME is None:
        SCENE_NAME = available_scenes[0]
    if SCENE_NAME not in available_scenes:
        raise RuntimeError(f'Scene {SCENE_NAME} not found in test split')

    subset_3d_root = Path(DRIVE_OPENSCENE_DATA_ROOT) / 'matterport_3d_one_scene'
    subset_3d_test = subset_3d_root / 'test'
    subset_fused_root = Path(DRIVE_OPENSCENE_DATA_ROOT) / 'matterport_multiview_openseg_test_one_scene'
    if subset_3d_root.exists():
        shutil.rmtree(subset_3d_root)
    if subset_fused_root.exists():
        shutil.rmtree(subset_fused_root)
    subset_3d_test.mkdir(parents=True, exist_ok=True)
    subset_fused_root.mkdir(parents=True, exist_ok=True)

    src_scene = DATASET_TEST_ROOT / f'{SCENE_NAME}.pth'
    dst_scene = subset_3d_test / src_scene.name
    shutil.copy2(src_scene, dst_scene)

    copied = 0
    for fused_file in FUSED_ROOT_FULL.glob(f'{SCENE_NAME}_*.pt'):
        shutil.copy2(fused_file, subset_fused_root / fused_file.name)
        copied += 1
    if copied == 0:
        raise RuntimeError(f'No fused feature files found for scene {SCENE_NAME}')

    DATA_ROOT = str(subset_3d_root)
    FUSED_ROOT = str(subset_fused_root)
    print('Using one-scene subset:', SCENE_NAME)
    print('Copied fused files:', copied)

os.environ['SAVE_ROOT'] = SAVE_ROOT
os.environ['TEST_REPEATS'] = str(TEST_REPEATS)
os.makedirs(SAVE_ROOT, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)
print('FUSED_ROOT =', FUSED_ROOT)
print('SAVE_ROOT =', SAVE_ROOT)
print('DATA_ROOT exists =', os.path.exists(DATA_ROOT))
print('FUSED_ROOT exists =', os.path.exists(FUSED_ROOT))


For the first run, keep `TEST_REPEATS=1` and disable visualization. Once this works on your full dataset or subset, increase repeats as needed.

In [ ]:
# Standalone long-running cell: safe to rerun after interruption.
from pathlib import Path

DRIVE_DATA_DIR = Path(DRIVE_OPENSCENE_DATA_ROOT)
DATASET_ROOT = DRIVE_DATA_DIR / 'matterport_3d'
DATASET_TEST_ROOT = DATASET_ROOT / 'test'
FUSED_ROOT_FULL = DRIVE_DATA_DIR / 'matterport_multiview_openseg_test'

DATA_ROOT = str(DATASET_ROOT)
FUSED_ROOT = str(FUSED_ROOT_FULL)
if BUILD_ONE_SCENE_SUBSET:
    available_scenes = sorted(p.stem for p in DATASET_TEST_ROOT.glob('*.pth'))
    if not available_scenes:
        raise RuntimeError('No test scenes found under matterport_3d/test')
    scene_name = SCENE_NAME or available_scenes[0]
    subset_3d_root = DRIVE_DATA_DIR / 'matterport_3d_one_scene'
    subset_fused_root = DRIVE_DATA_DIR / 'matterport_multiview_openseg_test_one_scene'
    subset_scene = subset_3d_root / 'test' / f'{scene_name}.pth'
    subset_feats = list(subset_fused_root.glob(f'{scene_name}_*.pt')) if subset_fused_root.exists() else []
    if not subset_scene.exists() or not subset_feats:
        raise RuntimeError('One-scene subset is missing or stale. Rerun the subset-build cell before eval.')
    DATA_ROOT = str(subset_3d_root)
    FUSED_ROOT = str(subset_fused_root)
    print('Evaluating one-scene subset:', scene_name)

%cd /content/repopt-3d/third_party/openscene
!python run/evaluate.py \
  --config=config/matterport/ours_openseg_pretrained.yaml \
  feature_type fusion \
  data_root {DATA_ROOT} \
  data_root_2d_fused_feature {FUSED_ROOT} \
  save_folder {SAVE_ROOT} \
  test_repeats {TEST_REPEATS} \
  vis_input False \
  vis_pred False \
  vis_gt False


In [ ]:
!ls -lah {SAVE_ROOT}
